# 00. Train Model - Locally

Helper notebook for training fire risk model locally.

Recommended for making sure model + data are correct. Training full model locally is time consuming

IMPORTANT: Currently, multi-dir training shuffles features and labels so that they are mismatched
Need to fix either in geebeam export, in an offline post-processing step, or at training time by joining
based on index.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import keras
import aic_risk_modeling as arm
import matplotlib.pyplot as plt


In [ ]:

all_stats = arm.train.load_stats_from_text('../../data/all_preds_emb_test_2023/stats.pbtxt')
arm.train.get_norm_stats(all_stats, 'BurnDate_0')

In [ ]:
# mcwd_stats = arm.train.load_stats_from_text('gs://aic-fire-amazon/data/terraclim_mcwd/stats.pbtxt')
# arm.train.get_norm_stats(mcwd_stats, 'mcwd_2022')
# veg_stats = arm.train.load_stats_from_text('gs://aic-fire-amazon/data/allpreds/stats.pbtxt')
# arm.train.get_norm_stats(veg_stats, 'EVI_begyear_2021')

In [ ]:
SEED = 54
RNG = np.random.default_rng(SEED)

# Set params

In [ ]:
DATA_DIRS = ["../../data/all_preds_emb_test_2020/", "../../data/all_preds_emb_test_2023/"]
TFRECORD_PATTERN='*.tfrecord.gz'
PATCH_SIZE=128
# For model, input and output bands
# INPUT_BANDS =["BurnDate"]
INPUT_BANDS =["A00", "A01", 'classification','mcwd','EVI_begyear','NDVI_begyear','EVI','NDVI','BurnDate']

OUTPUT_BANDS = ['BurnDate_0']

YEARS = [-3, -2, -1]

TRANSFORMS = {
        # "BurnDate": "gt0",
        # "BurnDate_0": "gt0_bool",
        "mcwd": "normalize_mcwd",
        "EVI": "normalize_evi",
        "EVI_begyear": "normalize_evi",
        "NDVI": "normalize_ndvi",
        "NDVI_begyear": "normalize_ndvi",
    }

In [ ]:
# Merged dataset test, merging along features-axis
training_ds = arm.train.build_merged_dataset(DATA_DIRS, TFRECORD_PATTERN, PATCH_SIZE, axis='features')

In [ ]:

training_ds = arm.train.data_loader.dataset_from_dir(
    DATA_DIRS[0],
    'training-*.tfrecord.gz',
    patch_size=PATCH_SIZE,
    batch_size=4,
    cache=False
    ).shuffle(8)
validation_ds = arm.train.data_loader.dataset_from_dir(
    DATA_DIRS[1],
    'training-*.tfrecord.gz',
    patch_size=PATCH_SIZE,
    batch_size=4,
    cache=False
    )

In [ ]:
# Take one batch to inspect keys
for batch in training_ds.take(1):
    batch_keys = set(batch.keys())

In [ ]:
# Select bands
training_ds = arm.train.data_loader.select_bands_transform(
    training_ds,
    input_bands=INPUT_BANDS,
    output_bands=OUTPUT_BANDS,
    transforms=TRANSFORMS,
    stack_time_series=True,
    stack_inputs=False,
    years=YEARS,
    include_coords=True

)
validation_ds = arm.train.data_loader.select_bands_transform(
    validation_ds,
    input_bands=INPUT_BANDS,
    output_bands=OUTPUT_BANDS,
    transforms=TRANSFORMS,
    stack_time_series=True,
    stack_inputs=False,
    years=YEARS,
    include_coords=True
)

In [ ]:
for inputs, labels in training_ds.take(7):
    print("Batch inputs keys:", list(inputs.keys()))
    print("Image shape:", inputs['image'].shape)
    print("Label shape:", labels.shape)
    print(np.unique(labels))
    print("Label dtype", labels.dtype)
    print("Input dtype", inputs['image'].dtype)
    # print(inputs['coords'])
    input_shape = inputs['image'].shape[1:] 
    plt.imshow(inputs['image'][0,1, :, :, 0])
    plt.show()
    # plt.imshow(inputs['image'][0,1, :, :, 1])
    # plt.show()
    # plt.imshow(inputs['image'][0,1, :, :, 2])
    # plt.show()
    # plt.imshow(inputs['image'][0,1, :, :, 3])
    # plt.show()
    # plt.imshow(inputs['image'][0,1, :, :, 4])
    plt.show()
    plt.imshow(labels[0])

In [ ]:
model = arm.train.get_convlstm(image_shape=input_shape, include_metadata=False, metadata_shape=(2,))
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0025),
    loss="Dice",
    metrics=[
        keras.metrics.BinaryIoU(target_class_ids=[1]),
        keras.metrics.AUC(),
    ]
    )

checkpoint_filepath = './checkpoint.model.keras'
model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    monitor='val_loss',
    mode='min',
    save_best_only=True)

early_stopping_callback = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    mode='min',
    patience=30)

model.fit(
    training_ds,
    validation_data=validation_ds,
    # class_weight={0:0.1, 1:0.9},
    epochs=25,
    callbacks=[model_checkpoint_callback, early_stopping_callback]
)

In [ ]:
new_model = tf.keras.models.load_model('checkpoint.model.keras')

In [ ]:
valid_masks = np.array([b[1][i].numpy() for b in training_ds for i in range(b[1].shape[0])])

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, jaccard_score

In [ ]:
out = new_model.predict(training_ds)

In [ ]:
valid_masks.max()

In [ ]:

print(f1_score(valid_masks.flatten()>0.5, out.flatten()>0.5))
print(recall_score(valid_masks.flatten()>0.5, out.flatten()>0.5))
print(precision_score(valid_masks.flatten()>0.5, out.flatten()>0.5))
print(jaccard_score(valid_masks.flatten()>0.5, out.flatten()>0.5))

In [ ]:
def visualize_risk_predict(input_batch, target_batch, output_i, batch_i, suptitle, cutoff=0.5):
    fig, axs = plt.subplots(1,3)
    fig.suptitle(suptitle)
    # Embeddings
    rgb = np.stack([
        input_batch['image'][batch_i, 2, :, :, 0].numpy(),
        input_batch['image'][batch_i, 2, :, :, 1].numpy(),
        input_batch['image'][batch_i, 2,:, :, 2].numpy()], axis=2)
    # shift
    vmin=-0.3
    vmax=0.3
    rgb = (rgb - vmin)/(vmax - vmin)
    axs.flatten()[0].imshow(rgb)
    axs.flatten()[0].set_title('Embeddings')


    # Prediction
    axs.flatten()[1].imshow(output_i)
    axs.flatten()[1].set_title('Predicted burned area 2024')

    # 2023 burn (target)
    axs.flatten()[2].imshow(target_batch[batch_i].numpy()>cutoff)
    axs.flatten()[2].set_title('Actual burned area 2024')
    fig.tight_layout()

    plt.show()


In [ ]:
import matplotlib.pyplot as plt
j = 0
for batch in training_ds:
    for i in range(batch[1].shape[0]):
        if (batch[1][i].numpy()>0.5).sum()>0 or (out[j]>0.5).sum()>0:
            visualize_risk_predict(
                input_batch = batch[0],
                target_batch = batch[1],
                output_i = out[j],
                batch_i=i,
                suptitle='Image {}'.format(j),
                cutoff=0.99
            )
        j+=1